<a href="https://colab.research.google.com/github/bioadex/weather-forecast-algorithm/blob/devops/weather_forecast_halle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Using Large Language Models (LLMs) for Temperature Forecasting

Forecasting time series data like temperature with LLMs is an emerging field, and it typically involves transforming the numerical time series into a text-based sequence that the LLM can process. Here are a few suggested approaches:

1.  **Textual Description and Generation**:
    *   **Convert Data to Text**: Represent historical temperature data as natural language descriptions. For example: "On 2024-12-28, the temperature was 5°C. On 2024-12-29, it was 6°C. On 2024-12-30, it was 4°C."
    *   **Prompt Engineering**: Ask the LLM to continue the pattern or predict future values: "Given the following temperature history: [insert historical data here], what will the temperature be for the next 31 days starting 2025-01-01? Provide the output in a structured format (e.g., date and temperature)."
    *   **Fine-tuning (if feasible)**: For more robust results, you might consider fine-tuning an LLM on a large dataset of time series patterns and their textual representations. This is resource-intensive but can yield better accuracy.

2.  **LLM as a Feature Extractor/Encoder for Traditional Models**:
    *   You could use the LLM to generate textual descriptions or embeddings from the time series data. These embeddings could then be fed into traditional time series forecasting models (e.g., ARIMA, Prophet, Neural Networks like LSTMs or Transformers) as additional features.

3.  **LLM for Explaining Forecasts**:
    *   Even if you use traditional models for the actual forecast, an LLM can be valuable for interpreting the results, explaining trends, or generating human-readable summaries of the forecast and its uncertainties.

4.  **Specialized LLM-based Models**:
    *   Research is ongoing into architectures that directly use transformer-like models (the basis of LLMs) for time series forecasting. These models might be designed to process numerical sequences directly or use novel tokenization methods for numerical data.

### Considerations:
*   **Data Representation**: How you convert your numerical temperature data into text is crucial. You might need to experiment with different formats (e.g., raw numbers, descriptive phrases, relative changes).
*   **Context Window**: LLMs have a limited context window. For longer forecasting horizons or if you need to incorporate a lot of historical data, you'll need strategies to summarize or select relevant past information.
*   **Numerical Accuracy**: LLMs are primarily designed for language generation, not precise numerical computation. Their ability to predict exact temperature values with high accuracy might be limited compared to purpose-built time series models.
*   **Prompt Engineering**: Crafting effective prompts is key to getting useful output from an LLM.

Given that your data ends on `2024-12-31 23:00`, you would need to prepare a prompt that clearly states the last known data point and asks for predictions for the next 31 days. I would recommend starting with approach 1 (Textual Description and Generation) due to its relative simplicity in implementation.

In [1]:
import pandas as pd

# Load the CSV data
file_path = '/content/weather_forecast_data_Halle.csv'
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
print(f"Successfully loaded data from {file_path}.")
display(df.head())

Successfully loaded data from /content/weather_forecast_data_Halle.csv.


,date,temperature_2m (°C),relative_humidity_2m (%),apparent_temperature (°C),precipitation (mm),rain (mm),surface_pressure (hPa),cloud_cover (%),wind_speed_100m (km/h),wind_direction_100m (°)
0,2024-01-01 00:00,6.2,74,2.9,0.0,0.0,993.1,100,22.7,226
1,2024-01-01 01:00,5.7,76,2.4,0.0,0.0,993.3,100,22.7,224
2,2024-01-01 02:00,5.6,77,2.3,0.0,0.0,993.1,98,22.7,221
3,2024-01-01 03:00,5.9,76,2.0,0.0,0.0,992.9,1,28.3,226
4,2024-01-01 04:00,5.8,77,1.3,0.0,0.0,993.1,19,33.1,224


In [3]:
df['date'] = pd.to_datetime(df['date'])

temperature_data_text = []
for index, row in df.iterrows():
    time_str = row['date'].strftime('%Y-%m-%d %H:%M')
    temperature = row['temperature_2m (°C)']
    temperature_data_text.append(f"on {time_str}, the temperature was {temperature} degree Celsius")

# Join the list into a single string for display or use with an LLM
full_text_representation = "; ".join(temperature_data_text)

# Display the first 5 entries of the textual representation as an example
print("Example of textual representation (first 5 entries):")
for i in range(min(5, len(temperature_data_text))):
    print(temperature_data_text[i])

# If you want to see the full text representation (be cautious with large datasets as it might be very long):
# print(full_text_representation)

Example of textual representation (first 5 entries):
on 2024-01-01 00:00, the temperature was 6.2 degree Celsius
on 2024-01-01 01:00, the temperature was 5.7 degree Celsius
on 2024-01-01 02:00, the temperature was 5.6 degree Celsius
on 2024-01-01 03:00, the temperature was 5.9 degree Celsius
on 2024-01-01 04:00, the temperature was 5.8 degree Celsius


### Prompt for Large Language Model (LLM) Forecasting

Now that the historical temperature data is in a textual format (`full_text_representation`), you can construct a prompt to feed into a Large Language Model (LLM) to forecast future temperatures.

Below is an example of such a prompt. You would typically pass this prompt to an LLM API or interface (e.g., Gemini, GPT, etc.).

In [6]:
# Construct the prompt for the LLM
prompt = f"""Given the following historical hourly temperature data:
{full_text_representation}

Continue this pattern and predict the future temperature (in degrees Celsius) for the next 31 days, starting from 2025-01-01 00:00. Provide the output in a structured format, listing each date and its predicted temperature like this:

YYYY-MM-DD HH:MM, X.X degree Celsius
YYYY-MM-DD HH:MM, Y.Y degree Celsius
...

Please ensure the predictions cover a full 31-day period, with hourly data points.
"""

print("--- LLM Prompt Example ---")
# Print a truncated version of the prompt to avoid overwhelming the output
print(prompt[:1000] + "...\n\n" + prompt[-500:])

--- LLM Prompt Example ---
Given the following historical hourly temperature data:
on 2024-01-01 00:00, the temperature was 6.2 degree Celsius; on 2024-01-01 01:00, the temperature was 5.7 degree Celsius; on 2024-01-01 02:00, the temperature was 5.6 degree Celsius; on 2024-01-01 03:00, the temperature was 5.9 degree Celsius; on 2024-01-01 04:00, the temperature was 5.8 degree Celsius; on 2024-01-01 05:00, the temperature was 5.6 degree Celsius; on 2024-01-01 06:00, the temperature was 6.0 degree Celsius; on 2024-01-01 07:00, the temperature was 6.7 degree Celsius; on 2024-01-01 08:00, the temperature was 6.6 degree Celsius; on 2024-01-01 09:00, the temperature was 5.8 degree Celsius; on 2024-01-01 10:00, the temperature was 5.8 degree Celsius; on 2024-01-01 11:00, the temperature was 6.1 degree Celsius; on 2024-01-01 12:00, the temperature was 6.7 degree Celsius; on 2024-01-01 13:00, the temperature was 7.4 degree Celsius; on 2024-01-01 14:00, the temperature was 7.7 degree Celsius; on

### Next Steps

1.  **Copy the `prompt` variable's content**: Take the full text generated in the `prompt` variable.
2.  **Use an LLM service**: Paste this prompt into your chosen LLM (e.g., Google's Gemini, OpenAI's GPT, Hugging Face models, etc.).
3.  **Analyze the LLM's response**: The LLM will generate text. You will need to parse its response to extract the predicted dates and temperatures. Since LLMs are generative, their output might not always be perfectly structured, so robust parsing will be necessary.
4.  **Evaluate predictions**: Compare the LLM's forecast against any available ground truth data or use other meteorological models for comparison.

### Using Large Language Models (LLMs) for Temperature Forecasting

Forecasting time series data like temperature with LLMs is an emerging field, and it typically involves transforming the numerical time series into a text-based sequence that the LLM can process. Here are a few suggested approaches:

1.  **Textual Description and Generation**:
    *   **Convert Data to Text**: Represent historical temperature data as natural language descriptions. For example: "On 2024-12-28, the temperature was 5°C. On 2024-12-29, it was 6°C. On 2024-12-30, it was 4°C."
    *   **Prompt Engineering**: Ask the LLM to continue the pattern or predict future values: "Given the following temperature history: [insert historical data here], what will the temperature be for the next 31 days starting 2025-01-01? Provide the output in a structured format (e.g., date and temperature)."
    *   **Fine-tuning (if feasible)**: For more robust results, you might consider fine-tuning an LLM on a large dataset of time series patterns and their textual representations. This is resource-intensive but can yield better accuracy.

2.  **LLM as a Feature Extractor/Encoder for Traditional Models**:
    *   You could use the LLM to generate textual descriptions or embeddings from the time series data. These embeddings could then be fed into traditional time series forecasting models (e.g., ARIMA, Prophet, Neural Networks like LSTMs or Transformers) as additional features.

3.  **LLM for Explaining Forecasts**:
    *   Even if you use traditional models for the actual forecast, an LLM can be valuable for interpreting the results, explaining trends, or generating human-readable summaries of the forecast and its uncertainties.

4.  **Specialized LLM-based Models**:
    *   Research is ongoing into architectures that directly use transformer-like models (the basis of LLMs) for time series forecasting. These models might be designed to process numerical sequences directly or use novel tokenization methods for numerical data.

### Considerations:
*   **Data Representation**: How you convert your numerical temperature data into text is crucial. You might need to experiment with different formats (e.g., raw numbers, descriptive phrases, relative changes).
*   **Context Window**: LLMs have a limited context window. For longer forecasting horizons or if you need to incorporate a lot of historical data, you'll need strategies to summarize or select relevant past information.
*   **Numerical Accuracy**: LLMs are primarily designed for language generation, not precise numerical computation. Their ability to predict exact temperature values with high accuracy might be limited compared to purpose-built time series models.
*   **Prompt Engineering**: Crafting effective prompts is key to getting useful output from an LLM.

Given that your data ends on `2024-12-31 23:00`, you would need to prepare a prompt that clearly states the last known data point and asks for predictions for the next 31 days. I would recommend starting with approach 1 (Textual Description and Generation) due to its relative simplicity in implementation.